# Proyecto final MEMAD – Semana 1

Implementación propia de optimizadores, validación contra PyTorch y experimento en funciones de prueba 2D.

**Cómo ejecutarlo:** menú *Entorno de ejecución → Ejecutar todas*. No necesita GPU (tarda unos 2 minutos en CPU).

Cada celda `%%writefile` crea un archivo `.py` en el entorno de Colab; las siguientes celdas lo ejecutan.

## 1. Implementación de los optimizadores (`optimizadores.py`)

In [ ]:
%%writefile optimizadores.py
# -*- coding: utf-8 -*-
"""
Implementación propia de algoritmos de optimización de primer orden.

Cada clase hereda de torch.optim.Optimizer y sigue las ecuaciones del
algoritmo original, con los mismos hiperparámetros por defecto que PyTorch
para poder validarlas contra torch.optim (ver test_optimizadores.py).

Notación: g_t = gradiente, theta = parámetros, eta = tasa de aprendizaje.
"""
import torch
from torch.optim import Optimizer


class SGDMomentum(Optimizer):
    """Descenso de gradiente estocástico con momento (heavy ball) o Nesterov.

        v_t     = mu * v_{t-1} + g_t                 (v_1 = g_1)
        clásico: theta_t = theta_{t-1} - eta * v_t
        Nesterov: theta_t = theta_{t-1} - eta * (g_t + mu * v_t)

    Con momentum=0 se reduce a SGD simple: theta_t = theta_{t-1} - eta * g_t.
    """

    def __init__(self, params, lr=1e-3, momentum=0.0, nesterov=False, weight_decay=0.0):
        if nesterov and momentum <= 0:
            raise ValueError("Nesterov requiere momentum > 0")
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov,
                                      weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, mu, wd = group['lr'], group['momentum'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if wd != 0:
                    g = g + wd * p                      # regularización L2
                if mu != 0:
                    state = self.state[p]
                    if 'v' not in state:
                        state['v'] = g.clone()
                    else:
                        state['v'].mul_(mu).add_(g)
                    v = state['v']
                    g = g + mu * v if group['nesterov'] else v
                p.add_(g, alpha=-lr)
        return loss


class Adagrad(Optimizer):
    """Adagrad (Duchi et al., 2011): tasa de aprendizaje adaptativa por parámetro.

        G_t     = G_{t-1} + g_t^2
        theta_t = theta_{t-1} - eta * g_t / (sqrt(G_t) + eps)
    """

    def __init__(self, params, lr=1e-2, eps=1e-10, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if group['weight_decay'] != 0:
                    g = g + group['weight_decay'] * p
                state = self.state[p]
                if 'G' not in state:
                    state['G'] = torch.zeros_like(p)
                state['G'].addcmul_(g, g)
                p.addcdiv_(g, state['G'].sqrt().add_(group['eps']), value=-group['lr'])
        return loss


class RMSprop(Optimizer):
    """RMSprop (Hinton, 2012): media móvil exponencial de g^2.

        s_t     = alpha * s_{t-1} + (1 - alpha) * g_t^2
        theta_t = theta_{t-1} - eta * g_t / (sqrt(s_t) + eps)
    """

    def __init__(self, params, lr=1e-2, alpha=0.99, eps=1e-8, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, alpha=alpha, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            alpha = group['alpha']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if group['weight_decay'] != 0:
                    g = g + group['weight_decay'] * p
                state = self.state[p]
                if 's' not in state:
                    state['s'] = torch.zeros_like(p)
                state['s'].mul_(alpha).addcmul_(g, g, value=1 - alpha)
                p.addcdiv_(g, state['s'].sqrt().add_(group['eps']), value=-group['lr'])
        return loss


class Adam(Optimizer):
    """Adam (Kingma & Ba, 2015) y AdamW (Loshchilov & Hutter, 2019).

        m_t     = beta1 * m_{t-1} + (1 - beta1) * g_t
        v_t     = beta2 * v_{t-1} + (1 - beta2) * g_t^2
        m_hat   = m_t / (1 - beta1^t)          (corrección de sesgo)
        v_hat   = v_t / (1 - beta2^t)
        theta_t = theta_{t-1} - eta * m_hat / (sqrt(v_hat) + eps)

    decoupled=False: weight decay como término L2 en el gradiente (Adam).
    decoupled=True:  weight decay aplicado directamente a los pesos (AdamW):
                     theta_{t-1} <- theta_{t-1} * (1 - eta * lambda).
    """

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=0.0, decoupled=False):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps,
                                      weight_decay=weight_decay, decoupled=decoupled))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, (b1, b2), eps, wd = group['lr'], group['betas'], group['eps'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if wd != 0:
                    if group['decoupled']:
                        p.mul_(1 - lr * wd)
                    else:
                        g = g + wd * p
                state = self.state[p]
                if 't' not in state:
                    state['t'] = 0
                    state['m'] = torch.zeros_like(p)
                    state['v'] = torch.zeros_like(p)
                state['t'] += 1
                t = state['t']
                state['m'].mul_(b1).add_(g, alpha=1 - b1)
                state['v'].mul_(b2).addcmul_(g, g, value=1 - b2)
                m_hat = state['m'] / (1 - b1 ** t)
                v_hat = state['v'] / (1 - b2 ** t)
                p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)
        return loss


def AdamW(params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2):
    """AdamW = Adam con weight decay desacoplado (mismo default que torch.optim.AdamW)."""
    return Adam(params, lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, decoupled=True)


## 2. Validación numérica contra `torch.optim`

Si todo está bien, las 9 filas deben decir **OK** (diferencias del orden de 1e-16).

In [ ]:
%%writefile test_optimizadores.py
# -*- coding: utf-8 -*-
"""
Validación numérica de los optimizadores propios contra torch.optim.

Para cada algoritmo se entrena la misma red pequeña (CNN), con la misma
inicialización y los mismos datos aleatorios, una vez con nuestra
implementación y otra con la de PyTorch. Si las ecuaciones son correctas,
los parámetros tras N pasos deben coincidir hasta el error de redondeo.

Se usa float64 para que la comparación sea estricta.
Uso:  python test_optimizadores.py
"""
import copy
import csv
import os
import sys

import torch
import torch.nn as nn

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import optimizadores as opt  # noqa: E402

torch.set_default_dtype(torch.float64)
N_STEPS = 200
TOL = 1e-10


def make_model():
    return nn.Sequential(
        nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(4, 8, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        nn.Flatten(), nn.Linear(8, 4),
    )


# (nombre, constructor propio, constructor de PyTorch)
CASOS = [
    ("SGD",
     lambda p: opt.SGDMomentum(p, lr=0.05),
     lambda p: torch.optim.SGD(p, lr=0.05)),
    ("SGD + momentum",
     lambda p: opt.SGDMomentum(p, lr=0.05, momentum=0.9),
     lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9)),
    ("Nesterov",
     lambda p: opt.SGDMomentum(p, lr=0.05, momentum=0.9, nesterov=True),
     lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9, nesterov=True)),
    ("SGD + momentum + L2",
     lambda p: opt.SGDMomentum(p, lr=0.05, momentum=0.9, weight_decay=1e-3),
     lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9, weight_decay=1e-3)),
    ("Adagrad",
     lambda p: opt.Adagrad(p, lr=0.05),
     lambda p: torch.optim.Adagrad(p, lr=0.05)),
    ("RMSprop",
     lambda p: opt.RMSprop(p, lr=1e-3),
     lambda p: torch.optim.RMSprop(p, lr=1e-3)),
    ("Adam",
     lambda p: opt.Adam(p, lr=1e-3),
     lambda p: torch.optim.Adam(p, lr=1e-3)),
    ("Adam + L2",
     lambda p: opt.Adam(p, lr=1e-3, weight_decay=1e-2),
     lambda p: torch.optim.Adam(p, lr=1e-3, weight_decay=1e-2)),
    ("AdamW",
     lambda p: opt.AdamW(p, lr=1e-3, weight_decay=1e-2),
     lambda p: torch.optim.AdamW(p, lr=1e-3, weight_decay=1e-2)),
]


def train(model, optimizer, X, y):
    losses = []
    crit = nn.CrossEntropyLoss()
    for i in range(N_STEPS):
        idx = slice((i * 16) % len(X), (i * 16) % len(X) + 16)   # mini-batches de 16
        optimizer.zero_grad()
        loss = crit(model(X[idx]), y[idx])
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses


def main():
    torch.manual_seed(0)
    X = torch.randn(256, 1, 16, 16)
    y = torch.randint(0, 4, (256,))
    base = make_model()

    filas, ok_total = [], True
    print(f"{'Optimizador':22s} {'max |Δθ|':>12s} {'|Δloss| final':>14s}  resultado")
    for nombre, propio, ref in CASOS:
        m1, m2 = copy.deepcopy(base), copy.deepcopy(base)
        l1 = train(m1, propio(m1.parameters()), X, y)
        l2 = train(m2, ref(m2.parameters()), X, y)
        diff = max((a - b).abs().max().item() for a, b in zip(m1.parameters(), m2.parameters()))
        dloss = abs(l1[-1] - l2[-1])
        ok = diff < TOL
        ok_total &= ok
        print(f"{nombre:22s} {diff:12.2e} {dloss:14.2e}  {'OK' if ok else 'FALLA'}")
        filas.append(dict(optimizador=nombre, max_diff_parametros=f"{diff:.3e}",
                          diff_loss_final=f"{dloss:.3e}", loss_inicial=f"{l1[0]:.4f}",
                          loss_final=f"{l1[-1]:.4f}", resultado='OK' if ok else 'FALLA'))

    out = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'resultados', 'validacion_optimizadores.csv')
    os.makedirs(os.path.dirname(out), exist_ok=True)
    with open(out, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(filas[0]))
        w.writeheader()
        w.writerows(filas)
    print(f"\n{N_STEPS} pasos por optimizador, tolerancia {TOL:g}. Resultados en {out}")
    sys.exit(0 if ok_total else 1)


if __name__ == '__main__':
    main()


In [ ]:
!python test_optimizadores.py

## 3. Experimento: trayectorias en funciones de prueba 2D

Cuadrática mal condicionada (κ = 20, rotada 30°) y función de Rosenbrock. Tarda 1–2 minutos.

In [ ]:
%%writefile trayectorias_2d.py
# -*- coding: utf-8 -*-
"""
Comportamiento de los optimizadores propios en funciones de prueba 2D.

  1. Cuadrática mal condicionada y rotada 30°:
         f(w) = 1/2 (w - w*)^T A (w - w*),  número de condición kappa = 20
  2. Función de Rosenbrock:
         f(x, y) = (1 - x)^2 + 100 (y - x^2)^2,  mínimo en (1, 1)

Para cada optimizador se elige la tasa de aprendizaje con una búsqueda en
rejilla logarítmica (la que da menor f tras el presupuesto de iteraciones),
de modo que la comparación sea justa. Se guardan las figuras y una tabla.

Uso:  python trayectorias_2d.py
"""
import csv
import math
import os
import sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import torch  # noqa: E402

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import optimizadores as opt  # noqa: E402

torch.set_default_dtype(torch.float64)
OUT = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'resultados')
os.makedirs(OUT, exist_ok=True)

OPTIMIZADORES = {
    'GD':             lambda p, lr: opt.SGDMomentum(p, lr=lr),
    'Momentum (0.9)': lambda p, lr: opt.SGDMomentum(p, lr=lr, momentum=0.9),
    'Nesterov (0.9)': lambda p, lr: opt.SGDMomentum(p, lr=lr, momentum=0.9, nesterov=True),
    'Adagrad':        lambda p, lr: opt.Adagrad(p, lr=lr),
    'RMSprop':        lambda p, lr: opt.RMSprop(p, lr=lr),
    'Adam':           lambda p, lr: opt.Adam(p, lr=lr),
}
COLORES = dict(zip(OPTIMIZADORES, ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']))
LR_GRID = np.logspace(-5, 1, 25)

# --- Funciones de prueba ----------------------------------------------------
KAPPA, ANG = 20.0, math.radians(30)
R = torch.tensor([[math.cos(ANG), -math.sin(ANG)], [math.sin(ANG), math.cos(ANG)]])
A = R @ torch.diag(torch.tensor([1.0, KAPPA])) @ R.T
W_STAR_Q = torch.tensor([0.0, 0.0])


def cuadratica(w):
    d = w - W_STAR_Q
    return 0.5 * d @ A @ d


def rosenbrock(w):
    return (1 - w[0]) ** 2 + 100 * (w[1] - w[0] ** 2) ** 2


PROBLEMAS = {
    'cuadratica': dict(f=cuadratica, x0=[-4.0, 3.0], iters=200, w_star=[0.0, 0.0],
                       xlim=(-5, 5), ylim=(-4, 4), titulo=f'Cuadrática rotada (κ = {KAPPA:.0f})'),
    'rosenbrock': dict(f=rosenbrock, x0=[-1.5, 2.0], iters=3000, w_star=[1.0, 1.0],
                       xlim=(-2, 2), ylim=(-1, 3), titulo='Rosenbrock'),
}


def run(f, x0, make_opt, lr, iters):
    w = torch.tensor(x0, requires_grad=True)
    o = make_opt([w], lr)
    path, vals = [w.detach().clone()], []
    for _ in range(iters):
        o.zero_grad()
        loss = f(w)
        vals.append(loss.item())
        loss.backward()
        o.step()
        if not torch.isfinite(w).all() or w.abs().max() > 1e6:
            return None, None
        path.append(w.detach().clone())
    vals.append(f(w).item())
    return torch.stack(path).numpy(), np.array(vals)


def mejor_lr(f, x0, make_opt, iters):
    mejor = (math.inf, None)
    for lr in LR_GRID:
        _, vals = run(f, x0, make_opt, lr, iters)
        if vals is not None and vals[-1] < mejor[0]:
            mejor = (vals[-1], lr)
    return mejor[1]


def main():
    filas = []
    for nombre_p, P in PROBLEMAS.items():
        f, x0, iters = P['f'], P['x0'], P['iters']
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2))

        xs = np.linspace(*P['xlim'], 300)
        ys = np.linspace(*P['ylim'], 300)
        Z = np.array([[f(torch.tensor([x, y])).item() for x in xs] for y in ys])
        ax1.contour(xs, ys, np.log10(Z + 1e-12), levels=30, cmap='Greys', linewidths=0.6)
        ax1.plot(*P['w_star'], 'k*', ms=14, label='Óptimo')
        ax1.plot(*x0, "ko", ms=6, label="Inicio")

        for nombre_o, make in OPTIMIZADORES.items():
            lr = mejor_lr(f, x0, make, iters)
            path, vals = run(f, x0, make, lr, iters)
            dist = np.linalg.norm(path - np.array(P['w_star']), axis=1)
            hit = np.where(dist < 1e-3)[0]
            filas.append(dict(problema=nombre_p, optimizador=nombre_o, lr=f"{lr:.2e}",
                              f_final=f"{vals[-1]:.3e}", dist_final=f"{dist[-1]:.3e}",
                              iter_dist_1e3=int(hit[0]) if len(hit) else 'no alcanza'))
            c = COLORES[nombre_o]
            ax1.plot(path[:, 0], path[:, 1], '-', color=c, lw=1.3, label=f'{nombre_o} (η={lr:.1e})')
            ax2.semilogy(np.maximum(vals, 1e-16), color=c, lw=1.5, label=nombre_o)
            print(f"{nombre_p:11s} {nombre_o:15s} lr={lr:.2e}  f_final={vals[-1]:.3e}  "
                  f"dist={dist[-1]:.2e}  iter(dist<1e-3)={filas[-1]['iter_dist_1e3']}")

        ax1.set_xlim(*P['xlim'])
        ax1.set_ylim(*P['ylim'])
        ax1.set_title(f"{P['titulo']}: trayectorias")
        ax1.legend(fontsize=8, loc='lower right' if nombre_p == 'rosenbrock' else 'upper right')
        ax2.set_title(f"{P['titulo']}: f(w) por iteración")
        ax2.set_xlabel('Iteración')
        ax2.set_ylabel('f(w)  (escala log)')
        ax2.grid(alpha=0.3)
        ax2.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(os.path.join(OUT, f'trayectorias_{nombre_p}.png'), dpi=130)
        plt.close(fig)

    with open(os.path.join(OUT, 'trayectorias_2d.csv'), 'w', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=list(filas[0]))
        w.writeheader()
        w.writerows(filas)
    print(f"\nFiguras y tabla guardadas en {OUT}")


if __name__ == '__main__':
    main()


In [ ]:
!python trayectorias_2d.py

## 4. Figuras y tablas de resultados

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(Image('resultados/trayectorias_cuadratica.png'))
display(Image('resultados/trayectorias_rosenbrock.png'))
display(pd.read_csv('resultados/validacion_optimizadores.csv'))
display(pd.read_csv('resultados/trayectorias_2d.csv'))

## 5. Descargar los resultados

Descarga un `.zip` con el código y los resultados para subirlo a Classroom.

In [ ]:
!zip -qr semana1.zip optimizadores.py test_optimizadores.py trayectorias_2d.py resultados
from google.colab import files
files.download('semana1.zip')